<a href="https://colab.research.google.com/github/kdimbu-spec/Intelligent-Agents-Portfolio/blob/main/AcademicResearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#The command installs all four libraries

### groq: Python client for interacting with the Groq AI API and running LLM-based applications.

### gradio: Creates simple web interfaces for AI models, demos, and applications.

### requests: Sends HTTP requests to APIs and websites to retrieve or send data.

### pandas: Data analysis and manipulation library using DataFrames.




In [365]:
!pip -q install groq gradio requests pandas

In [366]:
!pip -q install google-search-results

### Google Scholar Integration via SerpApi
To query Google Scholar programmatically, you can obtain a free API Key from [SerpApi](https://serpapi.com/). Add your API key below:

In [367]:
import os
from getpass import getpass

# Prompt the user to enter their SerpApi Key securely
os.environ["SERPAPI_API_KEY"] = getpass("Enter your SerpApi API Key: ")

Enter your SerpApi API Key: ··········


In [368]:
import os
from getpass import getpass

os.environ["GROQ_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [427]:
from groq import Groq

client = Groq(
    api_key=os.environ["GROQ_KEY"]
)

MODEL = "openai/gpt-oss-120b"

In [397]:
print('Available Groq models:')
for model in client.models.list().data:
    print(model.id)

Available Groq models:
whisper-large-v3-turbo
whisper-large-v3
canopylabs/orpheus-v1-english
qwen/qwen3.8-27b
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-safeguard-20b
allam-2-7b
openai/gpt-oss-20b
openai/gpt-oss-120b


In [371]:
from dataclasses import dataclass, field
from typing import List, Dict, Any

@dataclass
class ResearchState:

    research_question: str

    search_terms: List[str] = field(default_factory=list)

    raw_results: List[Dict[str, Any]] = field(default_factory=list)

    processed_results: List[Dict[str, Any]] = field(default_factory=list)

    report: str = ""

    retry_count: int = 0

    max_retries: int = 3

    sufficient: bool = False

    status: str = "INITIALISED"

    journal: List[str] = field(default_factory=list)

In [372]:
import json
import time
from groq import RateLimitError

def call_llm(prompt, temperature=0.2, max_retries=5):
    retries = 0
    delay = 1  # Initial delay in seconds

    while retries <= max_retries:
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": "You are an academic research planning agent."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=temperature
            )
            return response.choices[0].message.content
        except RateLimitError as e:
            retries += 1
            print(f"Rate limit exceeded. Retrying in {delay} seconds... (Attempt {retries}/{max_retries})")
            time.sleep(delay)
            delay *= 2  # Exponential backoff
            if retries > max_retries:
                raise e # Re-raise if max retries are exceeded
        except Exception as e:
            # Handle other potential errors without retrying (or implement specific retry logic for them)
            raise e


In [428]:
def planning_agent(state):

    prompt = f"""
You are the Planning Agent in a multi-agent academic
research system.

Research question:
{state.research_question}

Generate 5 precise academic search queries that can be
used to retrieve relevant scholarly literature.

The queries should cover:
1. The main research topic
2. Alternative terminology
3. Relevant methodology
4. Related theoretical concepts
5. Recent research

Return ONLY valid JSON in this format:

{{
    "queries": [
        "query 1",
        "query 2",
        "query 3",
        "query 4",
        "query 5"
    ]
}}
"""

    response = call_llm(prompt)

    try:
        data = json.loads(response)
        state.search_terms = data["queries"]

    except Exception:
        state.search_terms = [
            state.research_question
        ]

    state.journal.append(
        f"Planning Agent generated {len(state.search_terms)} search queries."
    )

    state.status = "PLANNING_COMPLETE"

    return state

In [429]:
from serpapi import GoogleSearch
import os

def search_google_scholar(query, per_page=5):
    """
    Queries Google Scholar using SerpApi.
    """
    api_key = os.environ.get("SERPAPI_API_KEY")
    if not api_key:
        print("Warning: SERPAPI_API_KEY is not set. Please provide a valid SerpApi Key.")
        return []

    params = {
        "engine": "google_scholar",
        "q": query,
        "hl": "en",
        "num": per_page,
        "api_key": api_key
    }

    try:
        search = GoogleSearch(params)
        results_dict = search.get_dict()
    except Exception as e:
        print(f"SerpApi Google Scholar Error: {e}")
        return []

    organic_results = results_dict.get("organic_results", [])

    results = []
    for item in organic_results:
        # Extract authors from publication information
        pub_info = item.get("publication_info", {})
        authors_list = []
        authors_data = pub_info.get("authors", [])
        if authors_data:
            authors_list = [author.get("name") for author in authors_data if author.get("name")]
        else:
            # Fallback text parsing for authors
            summary_text = pub_info.get("summary", "")
            if " - " in summary_text:
                authors_segment = summary_text.split(" - ")[0]
                authors_list = [a.strip() for a in authors_segment.split(",")]

        # Extract publication year
        year = None
        summary_text = pub_info.get("summary", "")
        for word in summary_text.replace(",", " ").split():
            if word.isdigit() and len(word) == 4:
                year = int(word)
                break

        results.append({
            "title": item.get("title"),
            "authors": authors_list,
            "year": year,
            "doi": None,
            "url": item.get("link"),
            "source": "Google Scholar"
        })

    return results

In [430]:
import xml.etree.ElementTree as ET
import requests

def search_arxiv(query, max_results=5):
    url = "http://export.arxiv.org/api/query"
    # Format query slightly to optimize search results on arXiv
    params = {
        "search_query": f"all:{query}",
        "max_results": max_results
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    # Parse Atom XML output
    root = ET.fromstring(response.text)
    # Namespace prefix handling for Atom format
    ns = {"atom": "http://www.w3.org/2005/Atom"}

    results = []
    for entry in root.findall("atom:entry", ns):
        title = entry.find("atom:title", ns)
        title_text = title.text.strip().replace("\n", " ") if title is not None else ""

        # Fetch authors
        authors = []
        for author in entry.findall("atom:author", ns):
            name = author.find("atom:name", ns)
            if name is not None:
                authors.append(name.text.strip())

        published = entry.find("atom:published", ns)
        year = int(published.text[:4]) if published is not None else None

        id_uri = entry.find("atom:id", ns)
        url_link = id_uri.text.strip() if id_uri is not None else ""

        results.append({
            "title": title_text,
            "authors": authors,
            "year": year,
            "doi": None,
            "url": url_link,
            "source": "arXiv"
        })

    return results

In [431]:
def retrieval_agent(state):
    all_results = []

    for query in state.search_terms:

        try:
            scholar_results = search_google_scholar(query, per_page=3)
            all_results.extend(scholar_results)
            state.journal.append(f"Retrieved {len(scholar_results)} results from Google Scholar for: {query}")
        except Exception as e:
            state.journal.append(f"Google Scholar retrieval error for '{query}': {str(e)}")

        try:
            arxiv_results = search_arxiv(query, max_results=3)
            all_results.extend(arxiv_results)
            state.journal.append(f"Retrieved {len(arxiv_results)} results from arXiv for: {query}")
        except Exception as e:
            state.journal.append(f"arXiv retrieval error for '{query}': {str(e)}")

    state.raw_results = all_results
    state.status = "RETRIEVAL_COMPLETE"
    return state

In [379]:
def deduplicate_results(results):

    unique = {}

    for item in results:

        title = item.get("title")

        if not title:
            continue

        key = title.lower().strip()

        if key not in unique:
            unique[key] = item

    return list(unique.values())

In [380]:
def processing_agent(state):

    state.raw_results = deduplicate_results(
        state.raw_results
    )

    state.journal.append(
        f"Deduplication completed. "
        f"{len(state.raw_results)} unique papers remain."
    )

    return state

In [381]:
def summarise_paper(paper):

    prompt = f"""
You are the Processing Agent in an academic research
system.

Analyse the following academic paper metadata.

Title:
{paper.get("title")}

Authors:
{", ".join(paper.get("authors", []))}

Year:
{paper.get("year")}

Provide:

1. Research relevance
2. Likely contribution
3. Methodology if identifiable
4. A short academic summary

Do not invent information that is not present.
"""

    return call_llm(prompt, temperature=0.1)

In [382]:
def process_papers(state, limit=10):

    processed = []

    for paper in state.raw_results[:limit]:

        summary = summarise_paper(paper)

        paper_copy = paper.copy()

        paper_copy["summary"] = summary

        processed.append(paper_copy)

    state.processed_results = processed

    state.journal.append(
        f"Processed {len(processed)} academic papers."
    )

    return state

In [384]:
from sklearn.feature_extraction.text import (
    TfidfVectorizer
)
from sklearn.metrics.pairwise import (
    cosine_similarity
)

def calculate_similarity(
    query,
    document
):

    vectorizer = TfidfVectorizer()

    vectors = vectorizer.fit_transform(
        [query, document]
    )

    score = cosine_similarity(
        vectors[0],
        vectors[1]
    )[0][0]

    return score

In [385]:
from datetime import datetime

def recency_score(year):

    current_year = datetime.now().year

    age = current_year - year

    score = max(
        0,
        1 - (age / 10)
    )

    return score

In [386]:
def citation_score(citations):

    max_citations = 1000

    return min(
        citations / max_citations,
        1.0
    )

In [425]:
def calculate_relevance(paper, question):
    text = f"{paper.get('title','')} {paper.get('summary','')}"
    return calculate_similarity(question, text)

In [387]:
def rank_results(state):

    for paper in state.processed_results:

        paper["relevance_score"] = calculate_relevance(
            paper,
            state.research_question
        )

    state.processed_results.sort(
        key=lambda x: x["relevance_score"],
        reverse=True
    )

    state.journal.append(
        "Research papers ranked by relevance."
    )

    return state

In [388]:
MIN_RESULTS = 8

In [389]:
def check_sufficiency(state):

    number_of_results = len(
        state.processed_results
    )

    if number_of_results >= MIN_RESULTS:

        state.sufficient = True
        state.status = "RESULTS_SUFFICIENT"

    else:

        state.sufficient = False
        state.status = "RESULTS_INSUFFICIENT"

    state.journal.append(
        f"Sufficiency check: "
        f"{number_of_results} results."
    )

    return state

In [390]:
def can_retry(state):

    return state.retry_count < state.max_retries

In [391]:
def retry_planning(state):

    state.retry_count += 1

    state.journal.append(
        f"Retry {state.retry_count} initiated."
    )

    return planning_agent(state)

In [392]:
def run_research(question):

    state = ResearchState(
        research_question=question
    )

    planning_agent(state)

    while True:

        retrieval_agent(state)

        processing_agent(state)

        # Reduce the limit of processed papers to avoid exceeding context window
        process_papers(state, limit=5)

        rank_results(state)

        check_sufficiency(state)

        # ----------------------------
        # DECISION NODE
        # ----------------------------

        if state.sufficient:

            break

        if not can_retry(state):

            state.status = (
                "MAX_RETRIES_REACHED"
            )

            break

        retry_planning(state)

    storage_agent(state)

    return state

In [432]:
def storage_agent(state):
    papers_text = ""

    # We limit to the top papers and truncate summaries to fit within Groq's 2,000 TPM limit
    for i, paper in enumerate(state.processed_results[:3], start=1):
        raw_summary = paper.get("summary", "")
        # Safely truncate summary to roughly 150 words to save tokens
        truncated_summary = " ".join(raw_summary.split()[:150]) + "..." if raw_summary else "No summary available."

        papers_text += f"""

Paper {i}
Title: {paper.get("title")}
Authors: {', '.join(paper.get("authors", []))}
Year: {paper.get("year")}
Source: {paper.get("source")}
URL: {paper.get("url")}

Summary:
{truncated_summary}
"""

    prompt = f"""
You are the Storage and Report Agent.

Research question:
{state.research_question}

Using the highly condensed research results below, produce a concise structured academic research report (keep your report brief to stay under the 2,000 token limit).

Include:
# Research Question
# Search Strategy
# Key Literature
# Synthesis of Findings
# Research Gaps
# Conclusion
# References

Research results:
{papers_text}
"""

    state.report = call_llm(
        prompt,
        temperature=0.1
    )

    state.status = "COMPLETED"
    state.journal.append(
        "Storage Agent generated final condensed research report safely within TPM limits."
    )

    return state

In [433]:
import json
def save_journal(state):

    journal_data = {
        "research_question":
            state.research_question,

        "search_terms":
            state.search_terms,

        "retry_count":
            state.retry_count,

        "status":
            state.status,

        "journal":
            state.journal,

        "results":
            state.processed_results
    }

    with open(
        "research_journal.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            journal_data,
            f,
            indent=2,
            ensure_ascii=False
        )

In [434]:
MIN_RESULTS = 3

question = """
How effective are machine learning techniques
in detecting COVID-19 misinformation on social
media platforms?
"""

state = run_research(question)

In [435]:
print(state.status)

COMPLETED


In [436]:
print("Retries:", state.retry_count)

Retries: 0


In [437]:
print("Papers:", len(state.processed_results))

Papers: 5


In [438]:
print(state.report)

**Research Report**

---

### 1. Research Question  
*How effective are machine‑learning (ML) techniques in detecting COVID‑19 misinformation on social‑media platforms?*  

---

### 2. Search Strategy  
| Step | Action | Sources & Keywords |
|------|--------|--------------------|
| 1 | Database selection | Google Scholar, ACM Digital Library, arXiv |
| 2 | Keyword string | “COVID‑19” AND “misinformation” AND “machine learning” AND (“Twitter” OR “social media”) |
| 3 | Inclusion criteria | Peer‑reviewed or pre‑print articles (2020‑2024), English, empirical ML models for textual COVID‑19 misinformation detection, focus on social‑media data. |
| 4 | Screening | Title/abstract review → full‑text assessment for methodological detail (dataset, model, evaluation). |
| 5 | Final set | 3 papers (Alenezi & Alqenaei 2021; Bojjireddy et al. 2021; Asher et al. 2023). |

---

### 3. Key Literature  

| Ref. | Platform & Data | ML Approach | Main Contribution |
|------|----------------|------------|-

In [439]:
!pip -q install gradio

In [441]:
import gradio as gr

In [442]:
def research_interface(question):

    if not question.strip():

        return (
            "Please enter a research question.",
            "",
            ""
        )

    state = run_research(question)

    report = state.report

    journal = "\n".join(
        state.journal
    )

    status = f"""
Status: {state.status}

Retries: {state.retry_count}

Papers processed:
{len(state.processed_results)}
"""

    return report, journal, status

In [443]:
with gr.Blocks(
    title="Academic Research Multi-Agent System"
) as Research_Agent:

    gr.Markdown(
        """
        # Academic Research Multi-Agent System

        An LLM-powered multi-agent system for
        academic literature discovery, processing
        and research report generation.
        """
    )

    question = gr.Textbox(
        label="Research Question",
        placeholder=(
            "Enter your academic research question..."
        ),
        lines=4
    )

    run_button = gr.Button(
        "Start Research"
    )

    status = gr.Textbox(
        label="System Status",
        lines=5
    )

    report = gr.Markdown(
        label="Research Report"
    )

    journal = gr.Textbox(
        label="Agent Activity Journal",
        lines=15
    )

    run_button.click(
        fn=research_interface,
        inputs=question,
        outputs=[
            report,
            journal,
            status
        ]
    )

Research_Agent.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b46b874e740c3eecfd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
